# SymPy justification for spectral regularity in the degenerate case


In [1]:
# import packages

import sympy as sp

sp.init_printing()


# introduce functions and notations
y, lam = sp.symbols("y lambda", real=True)
k, epsilon, iota = sp.symbols("k epsilon iota", real=True)
c = sp.symbols("c", nonzero=True)

b = sp.Function("b")(y)
f = sp.Function("f")(y, lam)
r = sp.Function("r")(y, lam)
chi = sp.Function("chi")(y)


b1 = sp.diff(b, y)
b2 = sp.diff(b, y, 2)
b3 = sp.diff(b, y, 3)
b4 = sp.diff(b, y, 4)
b5 = sp.diff(b, y, 5)
b6 = sp.diff(b, y, 6)

def D(expression):
    return sp.diff(expression, y) + b1 * sp.diff(expression, lam)

f_y = sp.diff(f, y)
f_lam = sp.diff(f, lam)
f_lam2 = sp.diff(f, lam, 2)
f_lam3 = sp.diff(f, lam, 3)
Df = D(f)
Df_y = sp.diff(Df, y)
D2f = D(Df)
D2f_y = sp.diff(D2f, y)
Dr = D(r)
D2r = D(Dr)
chi_y = sp.diff(chi, y)
chi_yy = sp.diff(chi, y, 2)

denominator = b - lam + sp.I * iota * epsilon
log_denominator = sp.log(denominator / c)

In [2]:
# define Rayleigh operator
def rayleigh_operator(expression):
    return (
        - sp.diff(expression, y, 2)
        + k**2 * expression
        + b2 * expression / denominator
    )

# Rayleigh equation for \partial_\lambda f
L_partial_lambda_f = (
    sp.diff(r, lam) - b2 * f / denominator**2
)

# Rayleigh equation for \partial_\lambda^2 f
L_partial_lambda_2_f = (
    sp.diff(r, lam, 2)
    - 2 * b2 * f_lam / denominator**2
    - 2 * b2 * f / denominator**3
)

# Rayleigh equation for \partial_\lambda D_\lambda f
L_partial_lambda_Df = (
    D(sp.diff(r, lam))
    - 2 * b2 * sp.diff(f_lam2, y)
    - b3 * f_lam2
    - b3 * f_lam / denominator
    - (b2 * Df + b3 * f) / denominator**2
)

# Rayleigh equation for \partial_lambda^3 f
L_partial_lambda_3_f = (
    sp.diff(r, lam, 3)
    - 3 * b2 * f_lam2 / denominator**2
    - 6 * b2 * f_lam / denominator**3
    - 6 * b2 * f / denominator**4
)

# Rayleigh equation for \partial_\lambda^2 D_\lambda f
L_partial_lambda_2_Df = (
    D(sp.diff(r, lam, 2)) - 2 * b2 * sp.diff(f_lam3, y)
    - b3 * f_lam3 - b3 * f_lam2 / denominator
    - 2 * (b2 * sp.diff(Df, lam) + b3 * f_lam) / denominator**2
    - 2 * (b2 * Df + b3 * f) / denominator**3
)

# Rayleigh equation for \partial_\lambda D_\lambda^2 f
L_partial_lambda_D2f = (
    D(D(sp.diff(r, lam))) - 4 * b2 * sp.diff(Df, lam, 2, y)
    - 2 * b3 * sp.diff(Df, lam, 2) + 2 * b2**2 * f_lam3
    - 2 * b3 * sp.diff(f_lam2, y) - b4 * f_lam2
    - (2 * b3 * sp.diff(Df, lam) + b4 * f_lam) / denominator
    - (b2 * D2f + 2 * b3 * Df + b4 * f) / denominator**2
)

In [3]:
# Rayleigh reductions

f_yy_reduction = k**2 * f + b2 * f / denominator - r

f_yyy_reduction = (
    k**2 * f_y - sp.diff(r, y)
    + (b3 * f + b2 * f_y) / denominator
    - b1 * b2 * f / denominator**2
)

f_yyyy_reduction = sp.expand(
    sp.diff(f_yyy_reduction, y).subs(sp.diff(f, y, 2), f_yy_reduction)
)

f_lamyy_reduction = (
    k**2 * f_lam
    + b2 * f_lam / denominator
    - sp.diff(r, lam)
    + b2 * f / denominator**2
)

f_lam2yy_reduction = (
    k**2 * f_lam2 + b2 * f_lam2 / denominator
    - sp.diff(r, lam, 2)
    + 2 * b2 * f_lam / denominator**2
    + 2 * b2 * f / denominator**3
)

f_lamyyy_reduction = sp.expand(sp.diff(f_lamyy_reduction, y))

f_lam3yy_reduction = (
    k**2 * f_lam3 + b2 * f_lam3 / denominator
    - sp.diff(r, lam, 3)
    + 3 * b2 * f_lam2 / denominator**2
    + 6 * b2 * f_lam / denominator**3
    + 6 * b2 * f / denominator**4
)

f_lam2yyy_reduction = sp.expand(sp.diff(f_lam2yy_reduction, y))

f_lamyyyy_reduction = sp.expand(sp.diff(f_yyyy_reduction, lam))

def apply_deg_rayleigh_reductions(expression):
    replacements = (
        (sp.diff(f, lam, y, 4), f_lamyyyy_reduction),
        (sp.diff(f, y, 4), f_yyyy_reduction),
        (sp.diff(f, lam, 2, y, 3), f_lam2yyy_reduction),
        (sp.diff(f, lam, y, 3), f_lamyyy_reduction),
        (sp.diff(f, lam, 3, y, 2), f_lam3yy_reduction),
        (sp.diff(f, lam, 2, y, 2), f_lam2yy_reduction),
        (sp.diff(f, lam, y, 2), f_lamyy_reduction),
        (sp.diff(f, y, 3), f_yyy_reduction),
        (sp.diff(f, y, 2), f_yy_reduction),
    )
    for old, new in replacements:
        expression = expression.subs(old, new)
    return sp.expand(expression)

## Justification for $\partial_\lambda f$:

In [4]:
# Set coefficients and construct S_{1,0}
S10 = - chi * f * b2 / b1**2 * log_denominator

In [5]:
# Calculate R_{1,0}(y,lambda) directly.
R10_direct = sp.expand(L_partial_lambda_f - rayleigh_operator(S10))

# apply reductions on R10_direct
R10_direct_reduced = apply_deg_rayleigh_reductions(R10_direct)

# display(sp.Eq(sp.Symbol("R10"), R10_direct))

In [6]:
# Hardcode R_{1,0}(y,lambda) from Appendix

R10_appendix_groups = {
    'order -2': (
        chi * log_denominator * (4*b2**2/b1**3*f_y - 6*b2**3/b1**4*f)
        + chi/denominator * (-2*b2/b1*f_y + 3*b2**2/b1**2*f)
        + chi_y*log_denominator * (-2*b2/b1**2*f_y + 4*b2**2/b1**3*f)
        + chi_y/denominator * (-2*b2/b1*f)
        + chi_yy*log_denominator * (-b2/b1**2*f)
    ),
    'order -1': (
        chi*log_denominator * (-2*b3/b1**2*f_y + 6*b2*b3/b1**3*f)
        + chi/denominator * (-2*b3/b1*f)
        + chi_y*log_denominator * (-2*b3/b1**2*f)
    ),
    'order 0': chi*log_denominator * (-b4/b1**2*f),
    'r terms': sp.diff(r, lam) + chi*log_denominator * (b2/b1**2*r),
    '1-chi terms': (1-chi)/denominator**2 * (-b2*f),
}
R10_appendix = sum(R10_appendix_groups.values(), sp.Integer(0))

# Apply reductions on R10_appendix.
R10_appendix_reduced = apply_deg_rayleigh_reductions(R10_appendix)

In [7]:
# Method 1: Use ``equals'' command
R10_direct_reduced.equals(R10_appendix_reduced)

True

In [8]:
# Method 2: calculate their difference.
sp.cancel(R10_direct_reduced - R10_appendix_reduced)

0

## Justification for $\partial_\lambda^2 f$:

In [9]:
# Set coefficients for S_{2,0}.
C1_20 = b2 / b1**2

C2_20 = -2 * b2 / b1**3

C3_20 = 3 * b2**2 / b1**4 - 2 * b3 / b1**3

# S_{2,0} construction
S20 = (
    C1_20 * f * chi/ denominator
    + (C2_20 * Df + C3_20 * f) * chi * log_denominator
)

In [10]:
# Calculate R_{2,0}(y,lambda) directly.
R20_direct = sp.expand(L_partial_lambda_2_f - rayleigh_operator(S20))

# apply reductions on R20_direct
R20_direct_reduced = apply_deg_rayleigh_reductions(R20_direct)

# display(sp.Eq(sp.Symbol("R20"), R20_direct))

In [11]:
R20_appendix_groups = {
    'order -4': (
        chi*log_denominator/denominator * (4*b2**3/b1**4*f)
        + chi*log_denominator * (
            8*b2**2/b1**4*Df_y - 20*b2**3/b1**5*Df
            - 28*b2**3/b1**5*f_y + 60*b2**4/b1**6*f
            + 4*k**2*b2**2/b1**4*f
        )
        + chi/denominator * (
            -4*b2/b1**2*Df_y + 10*b2**2/b1**3*Df
            + 2*b2**2/b1**3*f_y - 15*b2**3/b1**4*f
        )
        + chi_y*log_denominator * (
            -4*b2/b1**3*Df_y + 12*b2**2/b1**4*Df
            + 6*b2**2/b1**4*f_y - 24*b2**3/b1**5*f
        )
        + chi_y/denominator * (
            -4*b2/b1**2*Df + 2*b2/b1**2*f_y + 2*b2**2/b1**3*f
        )
        + chi_yy*log_denominator * (-2*b2/b1**3*Df + 3*b2**2/b1**4*f)
        + chi_yy/denominator * (b2/b1**2*f)
    ),
    'order -3': (
        chi*log_denominator/denominator * (-2*b2*b3/b1**3*f)
        + chi*log_denominator * (
            -4*b3/b1**3*Df_y + 16*b2*b3/b1**4*Df
            + 26*b2*b3/b1**4*f_y - 84*b3*b2**2/b1**5*f
        )
        + chi/denominator * (
            -4*b3/b1**2*Df - 2*b3/b1**2*f_y + 16*b2*b3/b1**3*f
        )
        + chi_y*log_denominator * (
            -4*b3/b1**3*Df - 4*b3/b1**3*f_y + 24*b2*b3/b1**4*f
        )
        + chi_y/denominator * (-2*b3/b1**2*f)
        + chi_yy*log_denominator * (-2*b3/b1**3*f)
    ),
    'order -2': (
        chi*log_denominator * (
            -2*b4/b1**3*Df - 4*b4/b1**3*f_y
            + 12*b3**2/b1**4*f + 18*b2*b4/b1**4*f
        )
        + chi/denominator * (-3*b4/b1**2*f)
        + chi_y*log_denominator * (-4*b4/b1**3*f)
    ),
    'order -1': chi*log_denominator * (-2*b5/b1**3*f),
    'r terms': (
        sp.diff(r, lam, 2)
        + chi*log_denominator * (2*b2/b1**3*Dr - 7*b2**2/b1**4*r + 2*b3/b1**3*r)
        + chi/denominator * (-b2/b1**2*r)
    ),
    '1-chi terms': (1-chi) * (-2*b2*f/denominator**3 - 2*b2*f_lam/denominator**2),
    'chi prime terms': chi_y/denominator**2 * (-2*b2/b1*f),
}
R20_appendix = sum(R20_appendix_groups.values(), sp.Integer(0))

# Apply reductions on R20_appendix.
R20_appendix_reduced = apply_deg_rayleigh_reductions(R20_appendix)

In [12]:
# Method 1: Use ``equals'' command
R20_direct_reduced.equals(R20_appendix_reduced)

True

In [13]:
# Method 2: calculate their difference.
sp.cancel(R20_direct_reduced - R20_appendix_reduced)

0

## Justification for $\partial_\lambda D_\lambda f$:

In [14]:
# Set coefficients for S_{1,1}.
C1_11 = -b2 / b1**2
C2_11 = -b3 / b1**2 + 2 * b2**2 / b1**3

# Construct S_{1,1}
S11 = (C1_11 * Df + C2_11 * f) * chi * log_denominator

In [15]:
# Calculate R_{1,1}(y,lambda) directly.
R11_direct = sp.expand(L_partial_lambda_Df - rayleigh_operator(S11))

# apply reductions on R11_direct
R11_direct_reduced = apply_deg_rayleigh_reductions(R11_direct)

# display(sp.Eq(sp.Symbol("R11"), R11_direct))

In [16]:
R11_appendix_groups = {
    'cancellations': -2*b2*sp.diff(f_lam2 - chi*b2*f/(b1**2*denominator), y),
    'order -3': (
        chi*log_denominator/denominator * (2*b2**3/b1**3*f)
        + chi*log_denominator * (
            2*b2**2/b1**3*Df_y - 4*b2**3/b1**4*Df
            - 14*b2**3/b1**4*f_y + 24*b2**4/b1**5*f
            + 2*k**2*b2**2/b1**3*f
        )
        + chi/denominator * (
            -2*b2/b1*Df_y + 3*b2**2/b1**2*Df
            + 2*b2**2/b1**2*f_y - 6*b2**3/b1**3*f
        )
        + chi_y*log_denominator * (
            -2*b2/b1**2*Df_y + 4*b2**2/b1**3*Df
            + 4*b2**2/b1**3*f_y - 12*b2**3/b1**4*f
        )
        + chi_y/denominator * (-2*b2/b1*Df + 2*b2**2/b1**2*f)
        + chi_yy*log_denominator * (-b2/b1**2*Df + 2*b2**2/b1**3*f)
    ),
    'order -2': (
        chi*log_denominator/denominator * (-b2*b3/b1**2*f)
        + chi*log_denominator * (
            -2*b3/b1**2*Df_y + 5*b2*b3/b1**3*Df
            + 13*b2*b3/b1**3*f_y - 36*b3*b2**2/b1**4*f
        )
        + chi/denominator * (-3*b3/b1*Df - b3/b1*f_y + 9*b2*b3/b1**2*f)
        + chi * (-b3*f_lam2)
        + (1-chi)/denominator * (-b3*f_lam)
        + (1-chi) * (-b3*f_lam2)
        + chi_y*log_denominator * (-2*b3/b1**2*Df - 2*b3/b1**2*f_y + 12*b2*b3/b1**3*f)
        + chi_y/denominator * (-2*b3/b1*f)
        + chi_yy*log_denominator * (-b3/b1**2*f)
    ),
    'order -1': (
        chi*log_denominator * (
            -b4/b1**2*Df - 2*b4/b1**2*f_y
            + 6*b3**2/b1**3*f + 8*b2*b4/b1**3*f
        )
        + chi/denominator * (-2*b4/b1*f)
        + chi_y*log_denominator * (-2*b4/b1**2*f)
    ),
    'order 0': chi*log_denominator * (-b5/b1**2*f),
    'r terms': D(sp.diff(r, lam)) + chi*log_denominator * (b2/b1**2*Dr - 4*b2**2/b1**3*r + b3/b1**2*r),
    '1-chi terms': (1-chi)/denominator**2 * (-b2*Df - b3*f),
}
R11_appendix = sum(R11_appendix_groups.values(), sp.Integer(0))

# Apply reductions on R11_appendix.
R11_appendix_reduced = apply_deg_rayleigh_reductions(R11_appendix)

In [17]:
# Method 1: Use ``equals'' command
R11_direct_reduced.equals(R11_appendix_reduced)

True

In [18]:
# Method 2: calculate their difference.
sp.cancel(R11_direct_reduced - R11_appendix_reduced)

0

## Justification for $\partial_\lambda^3f$:

In [19]:
# set coefficients for S_{3,0}
C1_30 = b2 / b1**2

C2_30 = -b2 / b1**3

C3_30 = 3 * b2 / b1**3

C4_30 = 2 * b3 / b1**3 - 3 * b2**2 / b1**4

C5_30 = -3 * b2 / b1**4

C6_30 = 12 * b2**2 / b1**5 - 6 * b3 / b1**4

C7_30 = (
    -k**2 * b2 / b1**4
    -3 * b4 / b1**4
    +16 * b2 * b3 / b1**5
    -15 * b2**3 / b1**6
)

C8_30 = sp.Integer(0)

# S_{3,0} construction
S30 = (
    chi * C1_30 * f / denominator**2
    + chi * (C2_30 * f_y + C3_30 * Df + C4_30 * f) / denominator
    + chi * (C5_30 * D2f + C6_30 * Df + C7_30 * f + C8_30) * log_denominator
)

In [20]:
# Calculate R_{3,0}(y,lambda) directly.
R30_direct = sp.expand(L_partial_lambda_3_f - rayleigh_operator(S30))

# apply reductions on R30_direct
R30_direct_reduced = apply_deg_rayleigh_reductions(R30_direct)

# display(sp.Eq(sp.Symbol("R30"), R30_direct))

In [21]:
R30_appendix_groups = {
    "order -6": (
        chi * log_denominator / denominator * (12*b2**3/b1**5*Df)
        + chi * log_denominator * (
            12*b2**2/b1**5*D2f_y - 48*b2**3/b1**6*D2f
            - 132*b2**3/b1**6*Df_y + 360*b2**4/b1**7*Df
            + 12*k**2*b2**2/b1**5*Df + 6*b2**3/b1**4*f_lam2
            + 48*b2**3/b1**5*sp.diff(f_y, lam)
            + 180*b2**4/b1**7*f_y + 8*k**2*b2**2/b1**5*f_y
            - 630*b2**5/b1**8*f - 20*k**2*b2**3/b1**6*f
        )
        + chi / denominator * (
            -6*b2/b1**3*D2f_y + 21*b2**2/b1**4*D2f
            + 12*b2**2/b1**4*Df_y - 78*b2**3/b1**5*Df
            - 12*b2**3/b1**5*f_y - 2*b2*k**2/b1**3*f_y
            + 105*b2**4/b1**6*f + 7*k**2*b2**2/b1**4*f
        )
        + chi_y * log_denominator * (
            -6*b2/b1**4*D2f_y + 24*b2**2/b1**5*D2f
            + 24*b2**2/b1**5*Df_y - 120*b2**3/b1**6*Df
            - 30*b2**3/b1**6*f_y - 2*b2*k**2/b1**4*f_y
            + 180*b2**4/b1**7*f + 8*k**2*b2**2/b1**5*f
        )
        + chi_y / denominator * (
            -6*b2/b1**3*D2f + 6*b2/b1**3*Df_y
            + 6*b2**2/b1**4*Df - 6*b2**3/b1**5*f
            - 4*b2*k**2/b1**3*f
        )
        + chi_yy * log_denominator * (
            -3*b2/b1**4*D2f + 12*b2**2/b1**5*Df
            - 15*b2**3/b1**6*f - b2*k**2/b1**4*f
        )
        + chi_yy / denominator * (
            3*b2/b1**3*Df - b2/b1**3*f_y - 3*b2**2/b1**4*f
        )
    ),
    "order -5": (
        chi * log_denominator / denominator * (
            -6*b2*b3/b1**4*Df + 24*b3*b2**2/b1**5*f
        )
        + chi * log_denominator * (
            -6*b3/b1**4*D2f_y + 30*b2*b3/b1**5*D2f
            + 102*b2*b3/b1**5*Df_y - 420*b3*b2**2/b1**6*Df
            - 18*b2*b3/b1**4*sp.diff(f_y, lam)
            + 24*b3*b2**2/b1**5*f_lam - 250*b3*b2**2/b1**6*f_y
            - 2*b3*k**2/b1**4*f_y + 1110*b3*b2**3/b1**7*f
            + 12*b2*b3*k**2/b1**5*f
        )
        + chi / denominator * (
            -6*b3/b1**3*D2f - 6*b3/b1**3*Df_y
            + 66*b2*b3/b1**4*Df + 14*b2*b3/b1**4*f_y
            - 150*b3*b2**2/b1**5*f - 4*b3*k**2/b1**3*f
        )
        + chi_y * log_denominator * (
            -6*b3/b1**4*D2f - 12*b3/b1**4*Df_y
            + 96*b2*b3/b1**5*Df + 32*b2*b3/b1**5*f_y
            - 250*b3*b2**2/b1**6*f - 2*b3*k**2/b1**4*f
        )
        + chi_y / denominator * (
            -6*b3/b1**3*Df
            + 2*b3/b1**3*f_y
            + 8*b2*b3/b1**4*f
        )
        + chi_yy * log_denominator * (
            -6*b3/b1**4*Df + 16*b2*b3/b1**5*f
        )
        + chi_yy / denominator * (2*b3/b1**3*f)
    ),
    "order -4": (
        chi * log_denominator / denominator * (
            -6*b3**2/b1**4*f - 3*b2*b4/b1**4*f
        )
        + chi * log_denominator * (
            -3*b4/b1**4*D2f - 12*b4/b1**4*Df_y
            + 48*b3**2/b1**5*Df + 72*b2*b4/b1**5*Df
            - 6*b3**2/b1**4*f_lam - 3*b2*b4/b1**4*f_lam
            + 32*b3**2/b1**5*f_y + 56*b2*b4/b1**5*f_y
            - 330*b2*b3**2/b1**6*f - 265*b4*b2**2/b1**6*f
            - b4*k**2/b1**4*f
        )
        + chi / denominator * (
            -9*b4/b1**3*Df - 3*b4/b1**3*f_y
            + 20*b3**2/b1**4*f + 35*b2*b4/b1**4*f
        )
        + chi_y * log_denominator * (
            -12*b4/b1**4*Df - 6*b4/b1**4*f_y
            + 32*b3**2/b1**5*f + 56*b2*b4/b1**5*f
        )
        + chi_y / denominator * (-2*b4/b1**3*f)
        + chi_yy * log_denominator * (-3*b4/b1**4*f)
    ),
    "order -3": (
        chi * log_denominator * (
            -6*b5/b1**4*Df - 6*b5/b1**4*f_y
            + 40*b2*b5/b1**5*f + 60*b3*b4/b1**5*f
        )
        + chi / denominator * (-4*b5/b1**3*f)
        + chi_y * log_denominator * (-6*b5/b1**4*f)
    ),
    "order -2": chi * log_denominator * (-3*b6/b1**4*f),
    "r-terms": (
        sp.diff(r, lam, 3)
        + chi * log_denominator * (
            3*b2/b1**4*D2r - 24*b2**2/b1**5*Dr
            + 6*b3/b1**4*Dr + 15*b2**3/b1**6*r
            + 3*b4/b1**4*r - 16*b2*b3/b1**5*r
            + b2*k**2/b1**4*r
        )
        + chi / denominator * (
            -3*b2/b1**3*Dr + b2/b1**3*sp.diff(r, y)
            + 3*b2**2/b1**4*r
        )
        + chi_y / denominator * (2*b2/b1**3*r)
    ),
    "1-chi terms": (1-chi) * (
        -6*b2*f/denominator**4 - 6*b2*f_lam/denominator**3
        - 3*b2*f_lam2/denominator**2
    ),
    "chi' terms": chi_y * (
        -4*b2/b1*f/denominator**3
        + (-6*b2/b1**2*Df + 4*b2/b1**2*f_y - 2*b3/b1**2*f)
        / denominator**2
    ),
    "chi'' terms": chi_yy * (b2/b1**2*f/denominator**2),
}
R30_appendix = sum(R30_appendix_groups.values(), sp.Integer(0))

# apply reductions on R30_appendix
R30_appendix_reduced = apply_deg_rayleigh_reductions(R30_appendix)

In [22]:
# Method 1: Use ``equals'' command
R30_direct_reduced.equals(R30_appendix_reduced)

True

In [23]:
# Method 2: calculate their difference.
sp.cancel(R30_direct_reduced - R30_appendix_reduced)

0

## Justification for $\partial_\lambda^2 D_\lambda f$:

In [24]:
# Set coefficients and construct S_{2,1}.
C1_21 = b2 / b1**2
C2_21 = b3 / b1**2 - 2 * b2**2 / b1**3
C3_21 = -2 * b2 / b1**3
C4_21 = 9 * b2**2 / b1**4 - 4 * b3 / b1**3
C5_21 = (
    12 * b2 * b3 / b1**4
    -12 * b2**3 / b1**5
    -2 * b4 / b1**3
)

# Construct S_{2,1}.
S21 = (
    chi * (C1_21 * Df + C2_21 * f) / denominator
    + chi * (C3_21 * D2f + C4_21 * Df + C5_21 * f) * log_denominator
)

In [25]:
# Calculate R_{2,1}(y, lambda) directly.
R21_direct = sp.expand(L_partial_lambda_2_Df - rayleigh_operator(S21))

# Apply the Rayleigh reductions.
R21_direct_reduced = apply_deg_rayleigh_reductions(R21_direct)

In [26]:
R21_appendix_groups = {
    "cancellations": (
        -2*b2*sp.diff(f_lam3-S30, y)
        -b3*(f_lam3-S30)
        -b3/denominator*(f_lam2-S20)
        -8*chi*b2**3/b1**3*f_lam2*log_denominator
    ),
    "order -5": (
        chi*log_denominator/denominator*(
            8*b2**3/b1**4*Df -22*b2**4/b1**5*f
            )
        +chi*log_denominator*(
            10*b2**2/b1**4*D2f_y -28*b2**3/b1**5*D2f
            -94*b2**3/b1**5*Df_y +250*b2**4/b1**6*Df
            +8*k**2*b2**2/b1**4*Df +28*b2**4/b1**5*f_lam
            +200*b2**4/b1**6*f_y +2*k**2*b2**2/b1**4*f_y
            -540*b2**5/b1**7*f -30*k**2*b2**3/b1**5*f
            )
        +chi/denominator*(
            -4*b2/b1**2*D2f_y +16*b2**2/b1**3*D2f
            +10*b2**2/b1**3*Df_y -63*b2**3/b1**4*Df
            -2*b2**3/b1**3*f_lam -12*b2**3/b1**4*f_y
            +90*b2**4/b1**5*f +2*k**2*b2**2/b1**3*f
            )
        +chi_y*log_denominator*(
            -4*b2/b1**3*D2f_y +18*b2**2/b1**4*D2f
            +18*b2**2/b1**4*Df_y -96*b2**3/b1**5*Df
            -24*b2**3/b1**5*f_y +150*b2**4/b1**6*f
            +2*k**2*b2**2/b1**4*f
            )
        +chi_y/denominator*(
            -4*b2/b1**2*D2f +2*b2/b1**2*Df_y
            +8*b2**2/b1**3*Df -2*b2**2/b1**3*f_y
            -6*b2**3/b1**4*f
            )
        +chi_yy*log_denominator*(
            -2*b2/b1**3*D2f +9*b2**2/b1**4*Df -12*b2**3/b1**5*f
            )
        +chi_yy/denominator*(b2/b1**2*Df -2*b2**2/b1**3*f)
    ),
    "order -4": (
        chi*log_denominator/denominator*(
            -2*b2*b3/b1**3*Df +26*b3*b2**2/b1**4*f
            )
        +chi*log_denominator*(
            -4*b3/b1**3*D2f_y +23*b2*b3/b1**4*D2f
            +64*b2*b3/b1**4*Df_y -285*b3*b2**2/b1**5*Df
            -22*b3*b2**2/b1**4*f_lam -251*b3*b2**2/b1**5*f_y
            +925*b3*b2**3/b1**6*f +15*b2*b3*k**2/b1**4*f
            )
        +chi/denominator*(
            -4*b3/b1**2*D2f -6*b3/b1**2*Df_y
            +52*b2*b3/b1**3*Df +2*b2*b3/b1**2*f_lam
            +12*b2*b3/b1**3*f_y -125*b3*b2**2/b1**4*f
            )
        +chi_y*log_denominator*(
            -4*b3/b1**3*D2f -8*b3/b1**3*Df_y
            +72*b2*b3/b1**4*Df +24*b2*b3/b1**4*f_y
            -200*b3*b2**2/b1**5*f
            )
        +chi_y/denominator*(
            -6*b3/b1**2*Df +2*b3/b1**2*f_y +8*b2*b3/b1**3*f
            )
        +chi_yy*log_denominator*(-4*b3/b1**3*Df +12*b2*b3/b1**4*f)
        +chi_yy/denominator*(b3/b1**2*f)
    ),
    "order -3": (
        chi*log_denominator/denominator*(
            -2*b3**2/b1**3*f -2*b2*b4/b1**3*f
            )
        +chi*log_denominator*(
            -2*b4/b1**3*D2f -8*b4/b1**3*Df_y
            +32*b3**2/b1**4*Df +48*b2*b4/b1**4*Df
            +4*b2*b4/b1**3*f_lam +28*b3**2/b1**4*f_y
            +48*b2*b4/b1**4*f_y -264*b2*b3**2/b1**5*f
            -212*b4*b2**2/b1**5*f
            )
        +chi/denominator*(
            -7*b4/b1**2*Df -2*b4/b1**2*f_y
            +16*b3**2/b1**3*f +28*b2*b4/b1**3*f
            )
        +chi_y*log_denominator*(
            -8*b4/b1**3*Df -4*b4/b1**3*f_y
            +24*b3**2/b1**4*f +42*b2*b4/b1**4*f
            )
        +chi_y/denominator*(-2*b4/b1**2*f)
        +chi_yy*log_denominator*(-2*b4/b1**3*f)
    ),
    "order -2": (
        chi*log_denominator*(
            -4*b5/b1**3*Df -4*b5/b1**3*f_y
            +30*b2*b5/b1**4*f +45*b3*b4/b1**4*f
            )
        +chi/denominator*(-3*b5/b1**2*f)
        +chi_y*log_denominator*(-4*b5/b1**3*f)
    ),
    "order -1": (
        chi*log_denominator*(-2*b6/b1**3*f)
    ),
    "r-terms": (
        D(sp.diff(r, lam, 2))
        +chi*log_denominator*(
            2*b2/b1**3*D2r -17*b2**2/b1**4*Dr
            +4*b3/b1**3*Dr +34*b2**3/b1**5*r
            +2*b4/b1**3*r -24*b2*b3/b1**4*r
            )
        +chi/denominator*(-b2/b1**2*Dr +2*b2**2/b1**3*r -b3/b1**2*r)
    ),
    "1-chi terms": (
        (1-chi)*(
            (-2*b2*Df -2*b3*f)/denominator**3
            +(-2*b2*sp.diff(Df, lam) -2*b3*f_lam)/denominator**2
            )
    ),
    "chi prime terms": (
        chi_y/denominator**2*(-2*b2/b1*Df +2*b2**2/b1**2*f -2*b3/b1*f)
    )
}

R21_appendix = sum(R21_appendix_groups.values(), sp.Integer(0))

# apply reductions on R30_appendix
R21_appendix_reduced = apply_deg_rayleigh_reductions(R21_appendix)

In [27]:
# Method 1: Use ``equals'' command
R21_direct_reduced.equals(R21_appendix_reduced)

True

In [28]:
# Method 2: calculate their difference.
sp.cancel(R21_direct_reduced - R21_appendix_reduced)

0

## Justification for $\partial_\lambda D_\lambda^2 f$:

In [29]:
# Set coefficients for S_{1,2}.
C1_12 = -b2 / b1**2
C2_12 = -2 * b3 / b1**2 + 4 * b2**2 / b1**3
C3_12 = -b4 / b1**2 + 6 * b2 * b3 / b1**3 - 6 * b2**3 / b1**4

# Construct S_{1,2}.
S12 = chi * (C1_12 * D2f + C2_12 * Df + C3_12 * f) * log_denominator

In [30]:
# Calculate R_{1,2}(y, lambda) directly.
R12_direct = sp.expand(L_partial_lambda_D2f - rayleigh_operator(S12))

# Apply the Rayleigh reductions.
R12_direct_reduced = apply_deg_rayleigh_reductions(R12_direct)

In [31]:
R12_appendix_groups = {
    "cancellations": (
        -4*b2*sp.diff(sp.diff(Df, lam, 2) - S21, y)
        +2*b2**2*(f_lam3 - S30)
        -2*b3*sp.diff(f_lam2 - S20, y)
    ),
    "order -4": (
        chi*log_denominator*(
            12*b2**2/b1**3*D2f_y - 36*b2**3/b1**4*D2f
            -60*b2**3/b1**4*Df_y + 216*b2**4/b1**5*Df
            -4*b2**2/b1**2*sp.diff(Df, y, lam)
            +2*b2**3/b1**2*f_lam2 + 8*b2**3/b1**3*sp.diff(f, y, lam)
            +96*b2**4/b1**5*f_y - 390*b2**5/b1**6*f
            -2*k**2*b2**3/b1**4*f
        )
        +chi/denominator*(
            -2*b2/b1*D2f_y + 11*b2**2/b1**2*D2f
            +4*b2**2/b1**2*Df_y - 42*b2**3/b1**3*Df
            -6*b2**3/b1**3*f_y + 60*b2**4/b1**4*f
        )
        +chi_y*log_denominator*(
            -2*b2/b1**2*D2f_y + 12*b2**2/b1**3*D2f
            +8*b2**2/b1**3*Df_y - 60*b2**3/b1**4*Df
            -12*b2**3/b1**4*f_y + 96*b2**4/b1**5*f
        )
        +chi_y/denominator*(
            -2*b2/b1*D2f + 4*b2**2/b1**2*Df - 4*b2**3/b1**3*f
        )
        +chi_yy*log_denominator*(
            -b2/b1**2*D2f + 4*b2**2/b1**3*Df - 6*b2**3/b1**4*f
        )
    ),
    "order -3": (
        chi*log_denominator/denominator*(
            -2*b2*b3/b1**2*Df + 4*b3*b2**2/b1**3*f
        )
        +chi*log_denominator*(
            -2*b3/b1**2*D2f_y + 14*b2*b3/b1**3*D2f
            +44*b2*b3/b1**3*Df_y - 216*b3*b2**2/b1**4*Df
            -2*b2*b3/b1**2*sp.diff(Df, lam)
            -6*b2*b3/b1**2*sp.diff(f, y, lam)
            +4*b3*b2**2/b1**3*f_lam
            -126*b3*b2**2/b1**4*f_y + 632*b3*b2**3/b1**5*f
        )
        +chi/denominator*(
            -2*b3/b1*D2f - 4*b3/b1*Df_y + 38*b2*b3/b1**2*Df
            -2*b3*sp.diff(Df, lam) + 6*b2*b3/b1**2*f_y
            -88*b3*b2**2/b1**3*f
        )
        +chi*(-2*b3*sp.diff(Df, lam, 2))
        +(1-chi)/denominator*(-2*b3*sp.diff(Df, lam))
        +(1-chi)*(-2*b3*sp.diff(Df, lam, 2))
        +chi_y*log_denominator*(
            -2*b3/b1**2*D2f - 4*b3/b1**2*Df_y
            +44*b2*b3/b1**3*Df + 12*b2*b3/b1**3*f_y
            -126*b3*b2**2/b1**4*f
        )
        +chi_y/denominator*(-4*b3/b1*Df + 6*b2*b3/b1**2*f)
        +chi_yy*log_denominator*(-2*b3/b1**2*Df + 6*b2*b3/b1**3*f)
    ),
    "order -2": (
        chi*log_denominator/denominator*(
            -2*b3**2/b1**2*f - b2*b4/b1**2*f
        )
        +chi*log_denominator*(
            -b4/b1**2*D2f - 4*b4/b1**2*Df_y
            +16*b3**2/b1**3*Df + 32*b2*b4/b1**3*Df
            -2*b3**2/b1**2*f_lam - b2*b4/b1**2*f_lam
            +16*b3**2/b1**3*f_y + 24*b2*b4/b1**3*f_y
            -162*b2*b3**2/b1**4*f - 138*b4*b2**2/b1**4*f
        )
        +chi/denominator*(
            -4*b4/b1*Df - b4*f_lam - 2*b4/b1*f_y
            +14*b3**2/b1**2*f + 19*b2*b4/b1**2*f
        )
        +chi*(-b4*f_lam2)
        +(1-chi)/denominator*(-b4*f_lam)
        +(1-chi)*(-b4*f_lam2)
        +chi_y*log_denominator*(
            -4*b4/b1**2*Df - 2*b4/b1**2*f_y
            +16*b3**2/b1**3*f + 24*b2*b4/b1**3*f
        )
        +chi_y/denominator*(-2*b4/b1*f)
        +chi_yy*log_denominator*(-b4/b1**2*f)
    ),
    "order -1": (
        chi*log_denominator*(
            -2*b5/b1**2*Df - 2*b5/b1**2*f_y
            +18*b2*b5/b1**3*f + 24*b3*b4/b1**3*f
        )
        +chi/denominator*(-2*b5/b1*f)
        +chi_y*log_denominator*(-2*b5/b1**2*f)
    ),
    "order 0": chi*log_denominator*(-b6/b1**2*f),
    "r-terms": (
        D(D(sp.diff(r, lam)))
        +chi*log_denominator*(
            b2/b1**2*D2r - 4*b2**2/b1**3*Dr
            +2*b3/b1**2*Dr + 6*b2**3/b1**4*r
            +b4/b1**2*r - 6*b2*b3/b1**3*r
        )
    ),
    "1-chi terms": (
        (1-chi)/denominator**2*(-b2*D2f - 2*b3*Df - b4*f)
    ),
}
R12_appendix = sum(R12_appendix_groups.values(), sp.Integer(0))

# Apply reductions on R12_appendix.
R12_appendix_reduced = apply_deg_rayleigh_reductions(R12_appendix)

In [32]:
# Method 1: Use ``equals'' command
R12_direct_reduced.equals(R12_appendix_reduced)

True

In [33]:
# Method 2: calculate their difference.
sp.cancel(R12_direct_reduced - R12_appendix_reduced)

0